# NLP Fundamentals

Core NLP concepts taught **step by step** with explicit code: preprocessing, representation, POS, NER, and sentiment analysis.

Project notebooks (`02`–`10`) reuse `nlp_helpers.py` once you are comfortable with the ideas here.


## 1. Introduction: What is NLP?

**Natural Language Processing** is a branch of AI that enables computers to understand, interpret, and generate human language. Key applications in professional settings include:

- **Customer Support**: Automatically route tickets, detect sentiment, summarize conversations
- **Content Moderation**: Flag inappropriate content, detect spam, identify fake reviews
- **Document Intelligence**: Extract key information from contracts, resumes, invoices
- **Search & Discovery**: Semantic search, recommendation systems, topic clustering
- **Business Intelligence**: Analyze feedback, monitor brand sentiment, trend detection

In [ ]:
import re
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Download NLTK data (run once per environment; safe to re-run)
nltk.download('punkt_tab', quiet=True)       # word_tokenize (NLTK 3.8+)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)  # pos_tag

# Tools we reuse in later sections
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

print('Libraries and NLTK data ready.')
print(f'English stopwords loaded: {len(stop_words)} words')


## 2. Text Preprocessing

Raw text is messy: inconsistent casing, punctuation, numbers, and noise. Preprocessing standardizes text so models can learn meaningful patterns.

### 2.1 Tokenization

**Tokenization** splits text into smaller units (tokens)—usually words or subwords. It's the first step in any NLP pipeline.

- **Word tokenization**: Splits on whitespace and punctuation ("Hello, world!" → ["Hello", ",", "world", "!"])
- **Sentence tokenization**: Splits into sentences
- **Subword tokenization**: Used in modern models (BERT, GPT) to handle rare words and morphologically rich languages

In [ ]:
# Tokenization examples
sample_text = "Customer reported: Order #4521 arrived damaged. Refund requested within 24 hours."

print("Original text:", sample_text)
print("\n--- Word Tokenization (NLTK) ---")
tokens = word_tokenize(sample_text)
print(tokens)

# Simple whitespace tokenization (common in quick scripts)
print("\n--- Simple split ---")
simple_tokens = sample_text.lower().split()
print(simple_tokens)

### 2.2 Lowercasing, Punctuation & Number Removal

These steps reduce vocabulary size and noise:
- **Lowercasing**: "Order" and "order" become the same token (unless case matters for your task, e.g., "Python" the language vs "python" the snake)
- **Punctuation removal**: Often doesn't add signal for bag-of-words models
- **Number removal**: Depends on the task—invoice numbers matter for some use cases, not for sentiment

In [ ]:
def basic_clean(text):
    """Basic text cleaning: lowercase, remove punctuation and numbers."""
    text = str(text).lower()
    # [^a-z\s]: ^ inside [] means "not"; match anything that is NOT a-z or whitespace
    # Replace each match with a single space (drops digits, punctuation, etc.)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # \s+: one or more whitespace chars (spaces, tabs, newlines) → single space
    text = re.sub(r'\s+', ' ', text).strip()  # strip() removes leading/trailing space
    return text

sample = "Order #4521 arrived DAMAGED! Refund requested within 24 hours."
print("Before:", sample)
print("After:", basic_clean(sample))

### 2.3 Stopwords

**Stopwords** are high-frequency words that typically add little semantic value (e.g., "the", "is", "at"). Removing them can:
- Reduce dimensionality
- Speed up training
- Sometimes improve performance (less noise)

**Caution**: For tasks like search ("to be or not to be") or sarcasm detection, stopwords can matter. Use domain judgment.

In [ ]:
# stop_words was created in the setup cell from nltk.corpus.stopwords
print('Sample stopwords:', sorted(list(stop_words))[:20])

def remove_stopwords(tokens):
    """Keep tokens that are not in the stopword list."""
    return [t for t in tokens if t.lower() not in stop_words]

text = 'The product quality is excellent and the delivery was fast.'
tokens = word_tokenize(text.lower())
print('Original tokens:', tokens)
print('After stopword removal:', remove_stopwords(tokens))


### 2.4 Stemming vs Lemmatization

Both reduce words to a base form to consolidate variations:

| Technique | How it works | Example | Use case |
|-----------|--------------|---------|----------|
| **Stemming** | Uses rules to chop suffixes | running → run, studies → studi | Fast, rough normalization |
| **Lemmatization** | Uses dictionary + morphology | running → run, studies → study | More accurate, needs POS for best results |

- **Stemming**: Faster, can produce invalid words ("studi")
- **Lemmatization**: Slower, produces valid dictionary words, better for interpretability

In [ ]:
# porter and lemmatizer were created in the setup cell
words = ['running', 'runs', 'ran', 'studies', 'studying', 'better', 'easily']

print('Word          | Stem (Porter) | Lemma (verb POS)')
print('-' * 48)
for w in words:
    stem = porter.stem(w)
    lemma = lemmatizer.lemmatize(w, pos='v')  # pos='v' tells WordNet: treat as verb
    print(f'{w:13} | {stem:13} | {lemma}')


### 2.5 Complete Preprocessing Pipeline

Here we chain the steps on one support ticket. Read each step in order—this is the same logic project notebooks wrap in `preprocess_text()` inside `nlp_helpers.py`.

| Step | What happens |
|------|----------------|
| 1 | Lowercase |
| 2 | `word_tokenize` (before stripping punctuation) |
| 3 | Remove non-letters per token |
| 4 | Drop stopwords and single-character tokens |
| 5 | Stem **or** lemmatize (with POS tags for lemmas) |


In [ ]:
def treebank_to_wordnet(tag):
    """Map Penn Treebank POS tag to WordNet POS for lemmatize()."""
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('N'):
        return wordnet.NOUN
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN  # default when unsure


ticket = (
    "Hi, I've been waiting for my REFUND for 2 weeks! "
    "The customer service is terrible. Please help!"
)
print('Original:', ticket)

# --- shared steps for both stem and lemma paths ---
text = str(ticket).lower()
tokens = word_tokenize(text)
# [^a-z]: remove digits/punctuation inside each token (e.g. refund!, 2)
tokens = [re.sub(r'[^a-z]', '', t) for t in tokens]
tokens = [t for t in tokens if t and t not in stop_words and len(t) > 1]
print('\nTokens after clean + stopword removal:', tokens)

# --- path A: Porter stemmer (fast, can create non-words) ---
stemmed = [porter.stem(t) for t in tokens]
print('\nStem (Porter): ', ' '.join(stemmed))

# --- path B: lemmatize with POS tags (slower, readable words) ---
pos_tags = nltk.pos_tag(tokens)
lemmatized = [
    lemmatizer.lemmatize(word, pos=treebank_to_wordnet(tag))
    for word, tag in pos_tags
]
print('Lemma (POS):   ', ' '.join(lemmatized))


## 3. Text Representation

Machine learning models need **numerical** input. We convert text into vectors through various schemes.

### 3.1 Bag of Words (BoW)

**Bag of Words** represents each document as a vector of word counts. It ignores word order and grammar—hence "bag".

- **Pros**: Simple, interpretable, works well for classification
- **Cons**: No semantics ("good" and "excellent" are separate), high dimensionality, sparse vectors

In [ ]:
# Bag of Words with sklearn
documents = [
    "Product quality is excellent",
    "Product quality is poor",
    "Excellent customer service"
]

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(documents)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nBoW matrix (documents x terms):")
print(pd.DataFrame(X_bow.toarray(), columns=vectorizer.get_feature_names_out()))

### 3.2 TF-IDF (Term Frequency-Inverse Document Frequency)

**TF-IDF** balances how important a term is to a document vs. how common it is across the corpus:

- **TF (Term Frequency)**: How often the term appears in the document
- **IDF (Inverse Document Frequency)**: Penalizes terms that appear in many documents (e.g., "the")

**Formula**: $\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{df}(t)}$

TF-IDF gives higher weight to distinctive terms (e.g., "refund" in a specific ticket) and lower weight to common terms.

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(documents)

print("TF-IDF matrix:")
print(pd.DataFrame(np.round(X_tfidf.toarray(), 3), columns=tfidf.get_feature_names_out()))

# Notice: "excellent" appears in 2 docs → lower IDF; "refund" would be high if it appeared in few docs

### 3.3 N-grams

**N-grams** are contiguous sequences of N tokens. Unigrams (1 word), bigrams (2 words), trigrams (3 words).

- **Why use n-grams?** Captures phrases: "not good" vs "good" has opposite meaning; "customer service" is one concept
- **Trade-off**: More n-grams → exponentially larger vocabulary → sparsity

In [ ]:
# Bigrams and trigrams
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))  # unigrams + bigrams
X_ngram = ngram_vectorizer.fit_transform(documents)

print("Features (unigrams + bigrams):")
print(ngram_vectorizer.get_feature_names_out())
print("\nSample: 'not good' as bigram captures negation!")

# Example where bigrams matter
neg_example = ["The product is not good", "The product is good"]
X_neg = ngram_vectorizer.fit_transform(neg_example)
print("\nBoW for 'not good' vs 'good':")
print(pd.DataFrame(X_neg.toarray(), columns=ngram_vectorizer.get_feature_names_out()))

## 4. Part-of-Speech (POS) Tagging

After tokenization, **POS tagging** labels each token with its grammatical role (noun, verb, adjective, etc.).

Why it matters:
- **Lemmatization** needs POS: `running` → `run` (verb) vs `running` → `running` (noun).
- **NER and parsing** pipelines use POS as an intermediate step.
- **Feature engineering**: e.g. count verbs in a review as a proxy for action-oriented language.

NLTK uses the **Penn Treebank** tag set. Common tags you'll see:

| Tag | Meaning | Example token |
|-----|---------|---------------|
| `NN` | Noun | `service` |
| `VB` | Verb, base form | `run` |
| `VBG` | Verb, gerund | `running` |
| `JJ` | Adjective | `terrible` |
| `NNP` | Proper noun | `Microsoft` |


In [ ]:
import nltk

sentence = "Apple announced a new product in San Francisco."

# Step 1: tokenize (POS taggers expect a list of tokens, not a raw string)
tokens = word_tokenize(sentence)

# Step 2: assign a tag to each (word, tag) pair
tagged = nltk.pos_tag(tokens)

print("Sentence:", sentence)
print("\nToken          | POS tag")
print("-" * 28)
for word, tag in tagged:
    print(f"{word:14} | {tag}")


### POS and lemmatization

In §2.5 we used `nltk.pos_tag` so `waiting` could lemmatize to `wait` (verb), not `waiting` (noun). Without POS, `WordNetLemmatizer` assumes **noun** by default.


In [ ]:
word = 'running'

# Default lemmatize() → assumes noun
print(f'{word} as noun (default): {lemmatizer.lemmatize(word)}')
# Explicit verb POS code used by WordNet
print(f'{word} as verb (pos="v"):  {lemmatizer.lemmatize(word, pos="v")}')

# Full sentence: same steps as §2.5
phrase = 'I am running late'
phrase_tokens = word_tokenize(phrase.lower())
phrase_tokens = [re.sub(r'[^a-z]', '', t) for t in phrase_tokens]
phrase_tokens = [t for t in phrase_tokens if t and t not in stop_words and len(t) > 1]
phrase_pos = nltk.pos_tag(phrase_tokens)
phrase_lemma = [
    lemmatizer.lemmatize(w, pos=treebank_to_wordnet(tag)) for w, tag in phrase_pos
]
print(f'Phrase lemmas: {" ".join(phrase_lemma)}')


## 5. Named Entity Recognition (NER)

**NER** finds **named entities** in text: people, organizations, locations, dates, etc.

Unlike POS (one label per token), entities are often **multi-word spans**:
- `San Francisco` → one location
- `Apple Inc.` → one organization

NLTK's `ne_chunk` builds on POS tags and groups tokens into labeled chunks. For larger projects, **spaCy** is the usual choice (`10-bbc-news-pos-ner-spacy.ipynb`).

| NLTK label | Typical meaning |
|------------|-----------------|
| `PERSON` | People |
| `ORGANIZATION` | Companies, agencies |
| `GPE` | Countries, cities, states |


In [ ]:
# NER needs extra models (not required for POS in §4)
nltk.download('maxent_ne_chunker_tab', quiet=True)  # ne_chunk (NLTK 3.8+)
nltk.download('words', quiet=True)
print('NER models ready.')


In [ ]:
def extract_entities(text):
    """Return (entity_type, entity_text) pairs using NLTK ne_chunk."""
    # Tokenize and tag — same first steps as POS
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)

    # ne_chunk groups consecutive tokens into entity subtrees
    chunks = nltk.ne_chunk(pos_tags)

    entities = []
    for chunk in chunks:
        # Plain tokens are strings; entity chunks are Tree objects with .label()
        if hasattr(chunk, 'label'):
            entity_text = ' '.join(word for word, _ in chunk)
            entities.append((chunk.label(), entity_text))
    return entities


news_line = (
    "Apple Inc. announced a deal with Microsoft in San Francisco on January 15, 2024."
)

print("Text:", news_line)
print("\nExtracted entities:")
for label, entity in extract_entities(news_line):
    print(f"  {label:14} {entity}")


## 6. Sentiment Analysis

**Sentiment analysis** predicts whether text expresses a positive, negative, or neutral attitude.

Common approaches:

| Approach | Idea | Strengths | Weaknesses |
|----------|------|-----------|------------|
| **Rule-based** | Count words from hand-built positive/negative lists | Simple, interpretable, no training data | Misses context (`not good`), slang, sarcasm |
| **Pre-trained lexicon** (e.g. VADER) | Scoring rules + curated sentiment lexicon | Handles negation, caps, emoticons better | Still weak on domain-specific jargon |
| **Supervised ML** | Train on labeled reviews (see `02-sentiment-restaurant-reviews.ipynb`) | Strong with enough data | Needs labels and maintenance |

Below we implement the first two **explicitly** so you see the logic before using library shortcuts.


### 6.1 Rule-based sentiment (custom word lists)

We tokenize, match tokens against small **positive** and **negative** sets, and compare counts.

```text
score = (# positive words) - (# negative words)
```

This is a toy version of what many production systems start with—and a useful baseline to beat.


In [ ]:
# Hand-curated lexicons (in practice you would expand and tune these for your domain)
POSITIVE_WORDS = {
    'good', 'great', 'excellent', 'love', 'happy', 'helpful',
    'fast', 'amazing', 'wonderful', 'recommend'
}
NEGATIVE_WORDS = {
    'bad', 'terrible', 'awful', 'hate', 'slow', 'rude',
    'poor', 'worst', 'disappointed', 'never'
}


def rule_based_sentiment(text):
    """
    Return (label, score) using simple positive-minus-negative word counts.
    """
    # Tokenize and keep alphabetic tokens only (same style as §2.5)
    tokens = word_tokenize(str(text).lower())
    tokens = [re.sub(r'[^a-z]', '', t) for t in tokens if t]

    pos_hits = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg_hits = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    score = pos_hits - neg_hits

    if score > 0:
        label = 'positive'
    elif score < 0:
        label = 'negative'
    else:
        label = 'neutral'
    return label, score, pos_hits, neg_hits


reviews = [
    'The food was excellent and the staff were very helpful.',
    'Slow service and rude staff. Never coming back.',
    'It was okay, nothing special.',
    'The food was not good.',  # rule-based usually fails here
]

print(f"{'Review':<55} | label    | score")
print('-' * 75)
for review in reviews:
    label, score, pos_hits, neg_hits = rule_based_sentiment(review)
    short = review[:52] + '...' if len(review) > 55 else review
    print(f'{short:<55} | {label:8} | {score:+d}  (+{pos_hits}/-{neg_hits})')


**Limitation:** `not good` still counts `good` as positive because we do not model negation. That is why pre-trained lexicons with linguistic rules—or supervised models—are often used next.


### 6.2 Pre-trained sentiment: VADER (NLTK)

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) ships with NLTK. It combines:

- A **pre-built sentiment lexicon** (words scored positive/negative)
- **Grammar rules** (e.g. `not` flips polarity, `very` intensifies, `!!!` boosts magnitude)

It returns four scores per sentence:

| Score | Meaning |
|-------|---------|
| `neg` | Proportion of negative |
| `neu` | Proportion of neutral |
| `pos` | Proportion of positive |
| `compound` | Single summary score in [-1, 1] (most common for decisions) |

Typical thresholds on `compound`: ≥ 0.05 → positive, ≤ -0.05 → negative, else neutral.


In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

# Pre-trained lexicon + rules (download once)
nltk.download('vader_lexicon', quiet=True)
vader = SentimentIntensityAnalyzer()

print('VADER ready.\n')


In [ ]:
def vader_sentiment(text, positive_threshold=0.05, negative_threshold=-0.05):
    """
    Return label and full VADER score dict.
    compound >= 0.05 → positive; <= -0.05 → negative; else neutral.
    """
    scores = vader.polarity_scores(text)
    compound = scores['compound']
    if compound >= positive_threshold:
        label = 'positive'
    elif compound <= negative_threshold:
        label = 'negative'
    else:
        label = 'neutral'
    return label, scores


# Same reviews as the rule-based example — compare approaches
print(f"{'Review':<50} | rule      | VADER     | compound")
print('-' * 90)
for review in reviews:
    rule_label, rule_score, _, _ = rule_based_sentiment(review)
    vader_label, vader_scores = vader_sentiment(review)
    short = review[:47] + '...' if len(review) > 50 else review
    print(
        f'{short:<50} | {rule_label:9} | {vader_label:9} | {vader_scores["compound"]:+.3f}'
    )

print('\nFull VADER breakdown for negation example:')
negation = 'The food was not good.'
print('Text:', negation)
print('Scores:', vader.polarity_scores(negation))


### Rule-based vs VADER — when to use what

- **Custom rules:** Full control, easy to audit, good for domain keywords (e.g. medical or legal terms).
- **VADER:** Strong out-of-the-box baseline for social media, reviews, and short text; still not perfect on sarcasm.
- **Supervised classifier (`02-…`):** Best when you have labeled data and care about your exact domain.

In practice, teams often start with VADER or a lexicon, then train TF-IDF + logistic regression or fine-tune transformers for production accuracy.


## 7. Next steps

You now have the core building blocks. Continue with project notebooks:

| Notebook | Topic |
|----------|--------|
| `02-sentiment-restaurant-reviews` | Supervised sentiment (TF-IDF + logistic regression) |
| `03`–`08` | Routing, search, LDA, keywords, spam |
| `09` | TripAdvisor preprocessing walkthrough |
| `06` | NER project (NLTK) |
| `10` | POS & NER with spaCy on BBC headlines |


## Summary

| Concept | Takeaway |
|---------|----------|
| **Preprocessing** | Tokenize first; clean per token; lemmatize with POS when possible. |
| **BoW / TF-IDF** | Turn text into numeric features for classical ML. |
| **N-grams** | Phrases matter—especially negation (`not good`). |
| **POS tagging** | Grammatical labels per token; feeds lemmatization and NER. |
| **NER** | Extract people, orgs, and places as multi-word spans. |
| **Sentiment** | Rule lists are simple; VADER adds pre-trained lexicon + rules; ML needs labels. |

Once these steps feel familiar, use `nlp_helpers.preprocess_text()` in project notebooks to avoid repeating boilerplate.
